# Product Price Predictor: From Frontier Baseline to QLoRA Fine-Tuning

End-to-end pipeline that:
1. Curates a synthetic product dataset using an LLM
2. Establishes a **Frontier model baseline** for price prediction
3. Analyzes the dataset for fine-tuning readiness
4. Prepares a complete **QLoRA fine-tuning config** for an open-source model
5. Evaluates and compares predictions

**Week 6 Community Contribution by Mirack**

Skills: Dataset curation, Frontier model inference, LoRA/QLoRA concepts, HuggingFace tokenizer analysis, evaluation metrics.

In [ ]:
import os
import json
import re
import math
from openai import OpenAI
from transformers import AutoTokenizer

In [ ]:
client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENAI_API_KEY")
)

MODEL = "gpt-4.1-nano"

## Step 1: Curate a Product Dataset
Generate realistic product descriptions with known prices across categories.

In [ ]:
def generate_products(category, num_products=5):
    """Generate realistic product listings with prices."""
    prompt = f"""Generate {num_products} realistic product listings for: {category}

Each product should have varied prices. Respond ONLY with a JSON array:
[
  {{
    "name": "Product Name",
    "description": "1-2 sentence description with brand, key features, specs",
    "category": "{category}",
    "price": 29.99
  }}
]

Make prices realistic. Include budget, mid-range, and premium options."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw)

In [ ]:
categories = ["Laptops", "Headphones", "Running Shoes", "Coffee Machines", "Backpacks"]
all_products = []

for cat in categories:
    print(f"Generating {cat}...")
    products = generate_products(cat, num_products=5)
    all_products.extend(products)

print(f"\nTotal products: {len(all_products)}")
print(f"Price range: ${min(p['price'] for p in all_products):.2f} - ${max(p['price'] for p in all_products):.2f}")

In [ ]:
# Preview the dataset
for p in all_products[:3]:
    print(f"[${p['price']:>8.2f}] {p['name']}")
    print(f"           {p['description']}\n")

## Step 2: Frontier Model Baseline
Use GPT to predict prices from descriptions alone — this is our benchmark to beat with fine-tuning.

In [ ]:
def predict_price_frontier(product_description, model=MODEL):
    """Use a Frontier model to predict a product's price."""
    prompt = f"""You are a pricing expert. Based on the product description below, predict the retail price in USD.
Respond with ONLY a number (e.g., 49.99). No text, no dollar sign.

Product: {product_description}"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"[^0-9.]", "", raw)
    try:
        return float(raw)
    except ValueError:
        return 0.0

In [ ]:
# Run predictions on all products
predictions = []

for p in all_products:
    desc = f"{p['name']}: {p['description']}"
    predicted = predict_price_frontier(desc)
    actual = p['price']
    error = abs(predicted - actual)
    predictions.append({
        "name": p['name'],
        "actual": actual,
        "predicted": predicted,
        "error": error,
        "pct_error": (error / actual * 100) if actual > 0 else 0
    })

print(f"{'Product':<35} {'Actual':>10} {'Predicted':>10} {'Error%':>8}")
print("-" * 65)
for p in predictions:
    print(f"{p['name'][:34]:<35} ${p['actual']:>8.2f} ${p['predicted']:>8.2f} {p['pct_error']:>7.1f}%")

In [ ]:
# Calculate evaluation metrics
errors = [p['error'] for p in predictions]
pct_errors = [p['pct_error'] for p in predictions]
squared_errors = [e**2 for e in errors]

mae = sum(errors) / len(errors)
rmse = math.sqrt(sum(squared_errors) / len(squared_errors))
mape = sum(pct_errors) / len(pct_errors)

print("=== Frontier Model Baseline Metrics ===")
print(f"MAE  (Mean Absolute Error):    ${mae:.2f}")
print(f"RMSE (Root Mean Square Error): ${rmse:.2f}")
print(f"MAPE (Mean Abs % Error):       {mape:.1f}%")
print(f"\nThese are the numbers to beat with a fine-tuned open-source model.")

## Step 3: Dataset Analysis for Fine-Tuning
Analyze token distributions to inform our QLoRA training config.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Format data as training prompts
training_prompts = []
for p in all_products:
    prompt = f"How much does this cost?\n\n{p['name']}: {p['description']}\n\nPrice is ${p['price']:.2f}"
    training_prompts.append(prompt)

token_lengths = [len(tokenizer.encode(t)) for t in training_prompts]

print(f"Training samples: {len(training_prompts)}")
print(f"Token stats:")
print(f"  Min: {min(token_lengths)} tokens")
print(f"  Max: {max(token_lengths)} tokens")
print(f"  Avg: {sum(token_lengths)/len(token_lengths):.0f} tokens")
print(f"  Total: {sum(token_lengths):,} tokens")
print(f"\nSample training prompt:")
print(training_prompts[0])

## Step 4: QLoRA Fine-Tuning Configuration
Complete setup ready to run on Google Colab with a GPU.

In [ ]:


qlora_config = {
    "base_model": "meta-llama/Llama-3.2-1B",
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "bfloat16",
        "bnb_4bit_use_double_quant": True
    },
    "lora": {
        "r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
        "lora_dropout": 0.05,
        "bias": "none",
        "task_type": "CAUSAL_LM"
    },
    "training": {
        "num_train_epochs": 3,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "warmup_ratio": 0.03,
        "lr_scheduler_type": "cosine",
        "max_seq_length": 256,
        "fp16": True
    }
}

print("=== QLoRA Fine-Tuning Configuration ===")
print(json.dumps(qlora_config, indent=2))

In [ ]:


colab_code = '''
# === Run this on Google Colab with a GPU runtime ===
# pip install transformers peft bitsandbytes trl datasets accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

# 1. Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# 2. Load base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# 3. LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 4. Training arguments
training_args = TrainingArguments(
    output_dir="./price_predictor_qlora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=10,
    save_strategy="epoch"
)

# 5. Train with SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,  # your formatted dataset
    args=training_args,
    max_seq_length=256
)
trainer.train()
model.save_pretrained("./price_predictor_adapter")
'''

print(colab_code)

## Step 5: Export Training Data
Format dataset for fine-tuning and save as JSONL.

In [ ]:
def format_for_finetuning(products):
    """Format products into chat-style training pairs."""
    training_data = []
    for p in products:
        entry = {
            "messages": [
                {"role": "user", "content": f"How much does this cost?\n\n{p['name']}: {p['description']}"},
                {"role": "assistant", "content": f"Price is ${p['price']:.2f}"}
            ]
        }
        training_data.append(entry)
    return training_data

training_data = format_for_finetuning(all_products)

print(f"Training samples ready: {len(training_data)}")
print(f"\nSample entry:")
print(json.dumps(training_data[0], indent=2))

## Step 6: LoRA vs QLoRA — Quick Comparison
Understanding the tradeoffs to pick the right approach.

In [ ]:
comparison = {
    "Full Fine-Tuning": {
        "trainable_params": "All (e.g., 1B for Llama-3.2-1B)",
        "memory": "~24GB+ VRAM",
        "speed": "Slowest",
        "quality": "Best (if enough data)",
        "hardware": "A100 / multi-GPU"
    },
    "LoRA": {
        "trainable_params": "~0.1-1% of total",
        "memory": "~12-16GB VRAM",
        "speed": "Fast",
        "quality": "Very close to full fine-tuning",
        "hardware": "T4 / L4 GPU"
    },
    "QLoRA": {
        "trainable_params": "~0.1-1% of total (4-bit base)",
        "memory": "~6-8GB VRAM",
        "speed": "Moderate (quantize/dequantize overhead)",
        "quality": "Comparable to LoRA",
        "hardware": "Free Colab T4!"
    }
}

print(f"{'Method':<20} {'Params':<30} {'Memory':<16} {'Hardware':<16}")
print("=" * 82)
for method, info in comparison.items():
    print(f"{method:<20} {info['trainable_params']:<30} {info['memory']:<16} {info['hardware']:<16}")

print(f"\nFor our {len(all_products)}-sample dataset, QLoRA with Llama-3.2-1B is the sweet spot.")
print("It runs on free Colab and can realistically compete with the Frontier baseline above.")